# Modelado e inferencia (MiBici) — Fase 2

Este notebook **no entrena ni registra modelos**: solo consume lo que generan los módulos.

1. `python -m mibici.features v3` genera el dataset `mibici_processed_v3`.
2. `python -m mibici.modeling.compare_models` evalúa las líneas base, Ridge y HGB Poisson con
   walk-forward, registra los runs en MLflow y deja al mejor modelo como `champion`.
3. Aquí se carga el dataset v3, se muestra la comparación registrada en MLflow y se predice
   con el `champion` mediante `mibici.modeling.predict`.

Toda la configuración se lee de `params.yaml`.

In [1]:
import os
import warnings

from loguru import logger

# Evitar rutas locales en las salidas: mensajes de import de mibici, la sugerencia de MLflow
# y el aviso de tqdm (sin ipywidgets)
logger.disable("mibici")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
warnings.filterwarnings("ignore", message="IProgress not found")

In [2]:
import mlflow
import pandas as pd

from mibici.config import MLFLOW_TRACKING_URI, PROCESSED_DATA_DIR, cargar_parametros
from mibici.modeling.predict import cargar_modelo, predecir
from mibici.modeling.validation import METRICAS

logger.enable("mibici")

params = cargar_parametros()

## 1. Dataset v3

In [3]:
datos = pd.read_parquet(PROCESSED_DATA_DIR / "estacion_hora_v3.parquet")

print(f"Dataset v3: {datos.shape[0]:,} filas x {datos.shape[1]} columnas")
print(f"Periodo: {datos['fecha_hora'].min()} a {datos['fecha_hora'].max()}")
datos.head()

Dataset v3: 3,445,646 filas x 21 columnas
Periodo: 2025-10-01 00:00:00 a 2026-09-30 22:00:00


,estacion_id,fecha_hora,hora,dia_semana,mes,fin_semana,salidas_lag_1,llegadas_lag_1,salidas_lag_24,llegadas_lag_24,...,llegadas_lag_168,festivo,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos,mes_sin,mes_cos,fecha_objetivo,salidas_siguiente_hora
0,2,2025-10-01 06:00:00,7.0,2.0,10.0,0.0,2.0,2.0,NaN,NaN,...,NaN,0.0,0.965926,-0.258819,0.974928,-0.222521,-1.0,-1.836970e-16,2025-10-01 07:00:00,5
1,2,2025-10-01 07:00:00,8.0,2.0,10.0,0.0,5.0,4.0,NaN,NaN,...,NaN,0.0,0.866025,-0.500000,0.974928,-0.222521,-1.0,-1.836970e-16,2025-10-01 08:00:00,4
2,2,2025-10-01 08:00:00,9.0,2.0,10.0,0.0,4.0,10.0,NaN,NaN,...,NaN,0.0,0.707107,-0.707107,0.974928,-0.222521,-1.0,-1.836970e-16,2025-10-01 09:00:00,5
3,2,2025-10-01 09:00:00,10.0,2.0,10.0,0.0,5.0,0.0,NaN,NaN,...,NaN,0.0,0.500000,-0.866025,0.974928,-0.222521,-1.0,-1.836970e-16,2025-10-01 10:00:00,3
4,2,2025-10-01 10:00:00,11.0,2.0,10.0,0.0,3.0,2.0,NaN,NaN,...,NaN,0.0,0.258819,-0.965926,0.974928,-0.222521,-1.0,-1.836970e-16,2025-10-01 11:00:00,2


## 2. Comparación de modelos (MLflow)

Métricas promedio de las ventanas walk-forward, tomadas de los runs padre que registra
`compare_models` en el experimento de `params.yaml`. Si un modelo se corrió varias veces se
muestra su run más reciente.

In [4]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

runs = mlflow.search_runs(
    experiment_names=[params["mlflow"]["experimento"]],
    order_by=["attributes.start_time DESC"],
)

# Runs padre de modelos: tienen métricas promedio y no tienen padre
columnas_metricas = [f"metrics.{nombre}_media" for nombre in METRICAS]
padres = runs[runs["tags.mlflow.parentRunId"].isna() & runs[columnas_metricas[0]].notna()]

comparacion = (
    padres.drop_duplicates("tags.mlflow.runName")
    .set_index("tags.mlflow.runName")[columnas_metricas]
    .rename_axis("modelo")
    .rename(columns=lambda columna: columna.removeprefix("metrics."))
    .sort_values(f"{next(iter(METRICAS))}_media")
)
comparacion

,mae_media,rmse_media,wape_media
modelo,,,
hgb_poisson,0.750155,1.309218,0.625581
baseline_media_historica,0.813038,1.517932,0.678746
ridge_ajustado,0.830844,1.432762,0.692822
baseline_lag_168,0.927024,1.856547,0.772841


## 3. Modelo champion

In [5]:
nombre_modelo = params["mlflow"]["modelo_registrado"]

version = mlflow.MlflowClient().get_model_version_by_alias(nombre_modelo, "champion")

print(f"{nombre_modelo}@champion -> versión {version.version} (run {version.run_id})")
pd.Series(version.tags, name="valor").to_frame()

mibici-demanda-salidas@champion -> versión 1 (run c91b95f79144492d884a3bc0c4250940)


,valor
algoritmo,hgb_poisson
horizonte_horas,1
rezagos,"1,24,168"
validacion,walk_forward
loss,poisson
learning_rate,0.1
max_iter,500
max_leaf_nodes,31
min_samples_leaf,100
l2_regularization,1.0


## 4. Inferencia con `mibici.modeling.predict`

Se predice el último mes de prueba de `params.yaml`. El `champion` se entrenó con todo el
periodo, así que estas métricas son **dentro de muestra**: sirven para comprobar que la carga
y la predicción funcionan, no para comparar modelos (eso es la sección 2).

In [6]:
modelo = cargar_modelo()

mes = pd.Period(params["validacion"]["mes_prueba_fin"], freq="M")
muestra = datos[datos["fecha_hora"].dt.to_period("M") == mes]

predicciones = predecir(muestra, modelo=modelo)

metricas = {
    nombre: metrica(muestra["salidas_siguiente_hora"], predicciones)
    for nombre, metrica in METRICAS.items()
}
print(f"{mes}: {len(muestra):,} predicciones | {metricas}")

resultado = muestra[["estacion_id", "fecha_hora", "salidas_siguiente_hora"]].copy()
resultado["prediccion"] = predicciones
resultado.head(10)

2026-11-05 14:01:29.753 | INFO     | mibici.modeling.predict:cargar_modelo:27 - Cargando modelo desde MLflow Model Registry: models:/mibici-demanda-salidas@champion


2026-09: 336,181 predicciones | {'mae': 0.8020065942910389, 'rmse': 1.390271766861525, 'wape': 0.5718502328284525}


,estacion_id,fecha_hora,salidas_siguiente_hora,prediccion
8034,2,2026-09-01 00:00:00,0,0.000870
8035,2,2026-09-01 01:00:00,0,0.000371
8036,2,2026-09-01 02:00:00,0,0.000263
8037,2,2026-09-01 03:00:00,0,0.000826
8038,2,2026-09-01 04:00:00,0,0.609012
8039,2,2026-09-01 05:00:00,3,0.956772
8040,2,2026-09-01 06:00:00,8,5.857443
8041,2,2026-09-01 07:00:00,3,8.593507
8042,2,2026-09-01 08:00:00,2,3.405963
8043,2,2026-09-01 09:00:00,3,2.979962
